In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import numpy as np
import os
from tqdm import tqdm
!pip install transformers -q
from transformers import ViTForImageClassification

# Setup
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

OUTPUT = '/kaggle/working/checkpoints'
os.makedirs(OUTPUT, exist_ok=True)

# Data transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
transform_vit = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
])

# Download CIFAR-10
print("Downloading CIFAR-10...")
full_train_cnn = torchvision.datasets.CIFAR10(
    root='/kaggle/working', train=True,  download=True, transform=transform_train)
full_train_vit = torchvision.datasets.CIFAR10(
    root='/kaggle/working', train=True,  download=True, transform=transform_vit)
test_set_cnn   = torchvision.datasets.CIFAR10(
    root='/kaggle/working', train=False, download=True, transform=transform_test)
test_set_vit   = torchvision.datasets.CIFAR10(
    root='/kaggle/working', train=False, download=True, transform=transform_vit)

# Fixed train/val split (45k/5k)
generator = torch.Generator().manual_seed(SEED)
train_cnn, val_cnn = torch.utils.data.random_split(
    full_train_cnn, [45000, 5000], generator=generator)
generator = torch.Generator().manual_seed(SEED)
train_vit, val_vit = torch.utils.data.random_split(
    full_train_vit, [45000, 5000], generator=generator)

# Data loaders
train_loader_cnn = torch.utils.data.DataLoader(train_cnn, batch_size=128, shuffle=True,  num_workers=2)
val_loader_cnn   = torch.utils.data.DataLoader(val_cnn,   batch_size=128, shuffle=False, num_workers=2)
train_loader_vit = torch.utils.data.DataLoader(train_vit, batch_size=64,  shuffle=True,  num_workers=2)
val_loader_vit   = torch.utils.data.DataLoader(val_vit,   batch_size=64,  shuffle=False, num_workers=2)
test_loader_cnn  = torch.utils.data.DataLoader(test_set_cnn, batch_size=128, shuffle=False, num_workers=2)
test_loader_vit  = torch.utils.data.DataLoader(test_set_vit, batch_size=64,  shuffle=False, num_workers=2)

print(f"Train: 45000 | Val: 5000 | Test: {len(test_set_cnn)}")

# Generic evaluation function
def evaluate(model, loader, is_vit=False):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(pixel_values=images).logits if is_vit else model(images)
            correct += outputs.argmax(1).eq(labels).sum().item()
            total   += labels.size(0)
    return correct / total

# Train CNN (ResNet-18)
print("\n" + "="*50)
print("TRAINING CNN (ResNet-18)")
print("="*50)

cnn_model = models.resnet18(weights='IMAGENET1K_V1')
cnn_model.fc = nn.Linear(cnn_model.fc.in_features, 10)
cnn_model = cnn_model.to(device)

optimizer_cnn = torch.optim.Adam(cnn_model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_cnn = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_cnn, T_max=15)
criterion     = nn.CrossEntropyLoss()

best_cnn_acc  = 0
cnn_history   = []

for epoch in range(15):
    cnn_model.train()
    running_loss = 0
    for images, labels in tqdm(train_loader_cnn, desc=f"CNN Epoch {epoch+1}/15"):
        images, labels = images.to(device), labels.to(device)
        optimizer_cnn.zero_grad()
        loss = criterion(cnn_model(images), labels)
        loss.backward()
        optimizer_cnn.step()
        running_loss += loss.item()
    scheduler_cnn.step()

    val_acc = evaluate(cnn_model, val_loader_cnn, is_vit=False)
    cnn_history.append({'epoch': epoch+1, 'val_acc': val_acc})
    print(f"  CNN Epoch {epoch+1}/15 — Loss: {running_loss/len(train_loader_cnn):.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_cnn_acc:
        best_cnn_acc = val_acc
        torch.save(cnn_model.state_dict(), f'{OUTPUT}/resnet18_best.pth')
        print(f"  --> New best CNN saved! ({val_acc:.4f})")

# Final test accuracy
cnn_model.load_state_dict(torch.load(f'{OUTPUT}/resnet18_best.pth'))
cnn_test_acc = evaluate(cnn_model, test_loader_cnn, is_vit=False)
print(f"\nCNN Best Val Acc: {best_cnn_acc:.4f}")
print(f"CNN Clean Test Acc: {cnn_test_acc:.4f}")

# Train ViT
print("\n" + "="*50)
print("TRAINING ViT")
print("="*50)

vit_model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=10,
    ignore_mismatched_sizes=True)
vit_model = vit_model.to(device)

# Only fine-tuning the classifier head + last 2 transformer blocks
# So training is faster and more stable
for name, param in vit_model.named_parameters():
    param.requires_grad = False
for name, param in vit_model.named_parameters():
    if any(x in name for x in ['classifier', 'encoder.layer.11', 'encoder.layer.10']):
        param.requires_grad = True

trainable = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in vit_model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

optimizer_vit = torch.optim.Adam(
    filter(lambda p: p.requires_grad, vit_model.parameters()),
    lr=2e-5, weight_decay=1e-4)
scheduler_vit = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_vit, T_max=10)

best_vit_acc = 0
vit_history  = []

for epoch in range(10):
    vit_model.train()
    running_loss = 0
    for images, labels in tqdm(train_loader_vit, desc=f"ViT Epoch {epoch+1}/10"):
        images, labels = images.to(device), labels.to(device)
        optimizer_vit.zero_grad()
        loss = criterion(vit_model(pixel_values=images).logits, labels)
        loss.backward()
        optimizer_vit.step()
        running_loss += loss.item()
    scheduler_vit.step()

    val_acc = evaluate(vit_model, val_loader_vit, is_vit=True)
    vit_history.append({'epoch': epoch+1, 'val_acc': val_acc})
    print(f"  ViT Epoch {epoch+1}/10 — Loss: {running_loss/len(train_loader_vit):.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_vit_acc:
        best_vit_acc = val_acc
        torch.save(vit_model.state_dict(), f'{OUTPUT}/vit_best.pth')
        print(f"  --> New best ViT saved! ({val_acc:.4f})")

# Final test accuracy
vit_model.load_state_dict(torch.load(f'{OUTPUT}/vit_best.pth'))
vit_test_acc = evaluate(vit_model, test_loader_vit, is_vit=True)
print(f"\nViT Best Val Acc: {best_vit_acc:.4f}")
print(f"ViT Clean Test Acc: {vit_test_acc:.4f}")

# print the summary
print("\n" + "="*50)
print("TRAINING COMPLETE")
print("="*50)
print(f"CNN  — Best Val: {best_cnn_acc:.4f} | Clean Test: {cnn_test_acc:.4f}")
print(f"ViT  — Best Val: {best_vit_acc:.4f} | Clean Test: {vit_test_acc:.4f}")
print(f"\nCheckpoints saved to: {OUTPUT}")
print("Files:")
for f in os.listdir(OUTPUT):
    size = os.path.getsize(f'{OUTPUT}/{f}')
    print(f"  {f}: {size/1e6:.1f} MB")